# Unstructured Data Pipeline — Real-World: Amazon Reviews 2023

This notebook uses the **Amazon Reviews 2023 dataset** (McAuley Lab, UCSD) as the real-world
unstructured corpus — replacing the synthetic GDELT news from `02_unstructured_pipeline.ipynb`.

Customer reviews are genuine unstructured text: varying length, colloquial language,
implicit sentiment, and diverse topics — making them an excellent stress-test for the
embedding + vector search + topic clustering pipeline.

## Dataset
**Source**: https://amazon-reviews-2023.github.io/  
**Categories used**: `Appliances`, `Electronics`, `Clothing_Shoes_and_Jewelry`  
**File format**: newline-delimited JSON, gzip-compressed (`.jsonl.gz`)

### Review record schema
```json
{
  "rating":            5.0,
  "title":             "Great product, works perfectly",
  "text":              "I bought this for my kitchen and it has been amazing...",
  "asin":              "B08XXXXXXX",
  "parent_asin":       "B08XXXXXXX",
  "user_id":           "ABCDEF123456",
  "timestamp":         1672531200000,
  "helpful_vote":      12,
  "verified_purchase": true
}
```

### How to download
Visit https://amazon-reviews-2023.github.io/ and download the review files for each category:
- `Appliances.jsonl.gz`
- `Electronics.jsonl.gz`
- `Clothing_Shoes_and_Jewelry.jsonl.gz`

Place them in `DATA_DIR` configured below.

## `AmazonReview` dataclass
This notebook defines its own `AmazonReview` dataclass (Step 3) with fields that match
the review domain directly. When the VectorDB backends are used, reviews are converted
to `NewsArticle` at that boundary only — the rest of the pipeline works with `AmazonReview`.

| `AmazonReview` field | Amazon Review source | Notes |
|----------------------|---------------------|-------|
| `id`                 | `user_id + asin` hash | Stable unique ID |
| `title`              | `title`             | Review summary (≤ 1024 chars) |
| `body`               | `text`              | Full review body (≤ 8192 chars) |
| `asin`               | `parent_asin`       | Product identifier |
| `category`           | loader-assigned     | Appliances / Electronics / Clothing |
| `rating`             | `rating`            | 1.0 – 5.0 stars |
| `sentiment_score`    | `(rating − 3) / 2`  | 1★=−1.0, 3★=0.0, 5★=+1.0 |
| `timestamp`          | `timestamp` (unix ms → ISO) | Actual review date |
| `embedding`          | Nemotron 1024-dim   | `passage: <title>. <body>` |
| `cluster_id`         | HDBSCAN label       | Discovered review topic |
| `cluster_label`      | Dominant keywords   | What reviewers talk about |


## Setup

In [ ]:
import gzip
import hashlib
import json
import logging
import os
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# ── Configuration ─────────────────────────────────────────────────────────────
# Update DATA_DIR to where you saved the .jsonl.gz files from:
# https://amazon-reviews-2023.github.io/
DATA_DIR = Path("./data/amazon_reviews").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_APP = Path("../app").resolve()
sys.path.insert(0, str(PROJECT_APP))

# Review JSONL.gz files — one per category
REVIEW_FILES = {
    "Appliances":              DATA_DIR / "Appliances.jsonl.gz",
    "Electronics":             DATA_DIR / "Electronics.jsonl.gz",
    "Clothing_Shoes_Jewelry":  DATA_DIR / "Clothing_Shoes_and_Jewelry.jsonl.gz",
}

MAX_REVIEWS_PER_CAT = 20_000   # cap per category — full files can be millions of rows
MIN_REVIEW_CHARS    = 50       # drop reviews shorter than this (low-signal)
VERIFIED_ONLY       = False    # True = keep only verified purchases

EMBEDDING_MODEL  = "nvidia/llama-3.2-nv-embedqa-1b-v2"
EMBED_DIM        = 1024
INFERENCE_BATCH  = 32
INGEST_BATCH     = 500
COLLECTION       = "amazon_reviews"
VECTORDB_BACKEND = "milvus"   # "milvus" or "elastic"
MILVUS_HOST      = "localhost"
MILVUS_PORT      = 19530
ELASTIC_HOST     = "localhost"
ELASTIC_PORT     = 9200
GPU_DEVICE       = 0

# GPU
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE = "cuda:0" if GPU_AVAILABLE else "cpu"
    print(f"PyTorch : {torch.__version__}  |  GPU: {'YES — ' + torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'NO (CPU)'}")
except ImportError:
    GPU_AVAILABLE, DEVICE = False, "cpu"
    print("PyTorch not installed")

print(f"Polars  : {pl.__version__}")
print(f"Data dir: {DATA_DIR}")
print()
for cat, path in REVIEW_FILES.items():
    size = f"{path.stat().st_size / 1e9:.2f} GB" if path.exists() else "NOT FOUND"
    print(f"  {cat:30s}: {path.name}  [{size}]")


---
## Step 1 — Load and sample review data from JSONL.gz

The review files are gzip-compressed newline-delimited JSON. We stream them line-by-line
and sample up to `MAX_REVIEWS_PER_CAT` per category to keep the notebook fast.

**Quality filters applied**:
- Minimum review body length (`MIN_REVIEW_CHARS`) — removes single-word reviews
- Optional `verified_purchase` filter
- Drop reviews with null `text` or `rating`

In [ ]:
def unix_ms_to_iso(ts_ms) -> str:
    """Convert unix millisecond timestamp to ISO-8601 string."""
    try:
        return datetime.fromtimestamp(int(ts_ms) / 1000, tz=timezone.utc).isoformat()
    except (TypeError, ValueError, OSError):
        return datetime.now(tz=timezone.utc).isoformat()


def stable_id(user_id: str, asin: str, idx: int) -> str:
    """Deterministic UUID from user + product + row index."""
    digest = hashlib.md5(f"{user_id}:{asin}:{idx}".encode()).hexdigest()
    return f"{digest[:8]}-{digest[8:12]}-{digest[12:16]}-{digest[16:20]}-{digest[20:32]}"


def load_reviews_jsonl_gz(
    path: Path,
    category: str,
    max_rows: int,
    min_chars: int = MIN_REVIEW_CHARS,
    verified_only: bool = VERIFIED_ONLY,
) -> list[dict]:
    """Stream a JSONL.gz review file and return up to max_rows clean records."""
    records: list[dict] = []
    seen = 0

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            seen += 1
            text    = (rec.get("text") or "").strip()
            rating  = rec.get("rating")
            verified = rec.get("verified_purchase", True)

            # Quality gates
            if not text or len(text) < min_chars:
                continue
            if rating is None:
                continue
            if verified_only and not verified:
                continue

            records.append({
                "category":   category,
                "id":         stable_id(rec.get("user_id", ""), rec.get("asin", ""), seen),
                "asin":       rec.get("parent_asin") or rec.get("asin", "UNKNOWN"),
                "title":      (rec.get("title") or "").strip()[:1024],
                "text":       text[:8192],
                "rating":     float(rating),
                "timestamp":  unix_ms_to_iso(rec.get("timestamp", 0)),
                "helpful_vote": int(rec.get("helpful_vote") or 0),
                "verified":   bool(verified),
            })

            if len(records) >= max_rows:
                break

    print(f"  [{category}] scanned {seen:,} lines → kept {len(records):,} reviews")
    return records


# ── Load all categories ────────────────────────────────────────────────────────
all_records: list[dict] = []
t0 = time.monotonic()

for category, path in REVIEW_FILES.items():
    if not path.exists():
        print(f"  [{category}] SKIPPED — file not found: {path}")
        print(f"    Download from: https://amazon-reviews-2023.github.io/")
        continue
    records = load_reviews_jsonl_gz(path, category, max_rows=MAX_REVIEWS_PER_CAT)
    all_records.extend(records)

print(f"\nTotal loaded: {len(all_records):,} reviews in {time.monotonic()-t0:.1f}s")

if not all_records:
    raise RuntimeError(
        "No reviews loaded. Download the datasets from:\n"
        "  https://amazon-reviews-2023.github.io/\n"
        "Then place the .jsonl.gz files in DATA_DIR and re-run."
    )

# ── Build Polars DataFrame for EDA ────────────────────────────────────────────
df_reviews = pl.DataFrame({
    "id":           [r["id"]           for r in all_records],
    "category":     [r["category"]     for r in all_records],
    "asin":         [r["asin"]         for r in all_records],
    "title":        [r["title"]        for r in all_records],
    "text":         [r["text"]         for r in all_records],
    "rating":       [r["rating"]       for r in all_records],
    "timestamp":    [r["timestamp"]    for r in all_records],
    "helpful_vote": [r["helpful_vote"] for r in all_records],
    "verified":     [r["verified"]     for r in all_records],
    "text_len":     [len(r["text"])    for r in all_records],
})

display(df_reviews.head(5))


---
## Step 2 — Exploratory analysis

Understand the distribution of ratings, review lengths, and category volume before
embedding — helps decide whether to sample uniformly or stratify.

In [ ]:
import matplotlib.pyplot as plt

print("=== Reviews by category ===")
display(
    df_reviews.group_by("category").agg([
        pl.len().alias("n_reviews"),
        pl.col("rating").mean().round(3).alias("avg_rating"),
        pl.col("rating").std().round(3).alias("std_rating"),
        pl.col("text_len").mean().round(0).alias("avg_text_len"),
        pl.col("helpful_vote").mean().round(2).alias("avg_helpful_votes"),
        pl.col("verified").mean().round(3).alias("pct_verified"),
    ]).sort("n_reviews", descending=True)
)

# Rating distribution per category
cats = df_reviews["category"].unique().to_list()
fig, axes = plt.subplots(1, len(cats), figsize=(6 * len(cats), 4), sharey=True)
if len(cats) == 1:
    axes = [axes]

for ax, cat in zip(axes, sorted(cats)):
    ratings = df_reviews.filter(pl.col("category") == cat)["rating"].to_list()
    star_counts = Counter(int(r) for r in ratings)
    stars  = [1, 2, 3, 4, 5]
    counts = [star_counts.get(s, 0) for s in stars]
    colors = ["#d32f2f", "#ef6c00", "#f9a825", "#7cb342", "#2e7d32"]
    ax.bar([str(s) + "★" for s in stars], counts, color=colors, alpha=0.85)
    ax.set_title(f"{cat}\n({len(ratings):,} reviews)")
    ax.set_xlabel("Star Rating")
    ax.grid(alpha=0.3, axis="y")

axes[0].set_ylabel("Review Count")
fig.suptitle("Rating Distribution by Category", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / "rating_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

# Review text length distribution
fig, ax = plt.subplots(figsize=(10, 4))
for cat in sorted(cats):
    lengths = df_reviews.filter(pl.col("category") == cat)["text_len"].to_list()
    ax.hist(np.log10(np.clip(lengths, 1, None)), bins=50, alpha=0.5, label=cat)
ax.set_xlabel("log₁₀(Review Length in chars)")
ax.set_ylabel("Count")
ax.set_title("Review Text Length Distribution")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "review_length_dist.png", dpi=120)
plt.show()


---
## Step 3 — Define `AmazonReview` and map review records

We define `AmazonReview` here in the notebook — a dataclass whose fields match the
review domain naturally. No more `headline` or `symbol` — just `title`, `body`, `asin`,
`category`, and `rating`.

**Sentiment mapping**: `(rating − 3) / 2.0`
- 1★ → −1.0 (very negative)  
- 2★ → −0.5  
- 3★ →  0.0 (neutral)  
- 4★ → +0.5  
- 5★ → +1.0 (very positive)

**Document text** = `"<title>. <body>"` — both title and body are embedded together so
queries on topics match both summary and detailed text.

> When we talk to the VectorDB (Milvus/Elasticsearch) later, we convert `AmazonReview`
> to `NewsArticle` at that boundary only — everything else in this notebook uses
> `AmazonReview` directly.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class AmazonReview:
    """Pipeline record for a single Amazon customer review."""
    id:              str
    title:           str           # review summary
    body:            str           # full review text
    asin:            str           # product identifier (parent_asin)
    category:        str           # Appliances | Electronics | Clothing_Shoes_Jewelry
    rating:          float         # 1.0 – 5.0 stars
    sentiment_score: float         # (rating − 3) / 2  →  [-1.0, +1.0]
    timestamp:       str           # ISO-8601 review date
    helpful_vote:    int  = 0
    verified:        bool = True
    embedding:       list[float] | None = None
    cluster_id:      int  = -1
    cluster_label:   str  = ""
    x:               float = 0.0
    y:               float = 0.0


def rating_to_sentiment(rating: float) -> float:
    return round(float(np.clip((rating - 3.0) / 2.0, -1.0, 1.0)), 4)


reviews: list[AmazonReview] = []
review_texts: list[str] = []   # combined title+body passed to the embedding model

for rec in all_records:
    doc = f"{rec['title']}. {rec['text']}".strip(" .")

    reviews.append(
        AmazonReview(
            id=rec["id"],
            title=rec["title"] or rec["text"][:120],
            body=rec["text"],
            asin=rec["asin"],
            category=rec["category"],
            rating=rec["rating"],
            sentiment_score=rating_to_sentiment(rec["rating"]),
            timestamp=rec["timestamp"],
            helpful_vote=rec["helpful_vote"],
            verified=rec["verified"],
            embedding=None,        # filled in Step 4
            cluster_id=-1,
            cluster_label=rec["category"],  # initial label = category
        )
    )
    review_texts.append(doc)

sentiments = np.array([r.sentiment_score for r in reviews])

print(f"Built {len(reviews):,} AmazonReview objects")
print(f"Sentiment range: [{sentiments.min():.2f}, {sentiments.max():.2f}]  "
      f"mean={sentiments.mean():.3f}")
print(f"Avg review length: {np.mean([len(t) for t in review_texts]):.0f} chars")
print(f"\nSample review text:\n  {review_texts[0][:200]}...")

# Sentiment distribution by category
df_sentiment = pl.DataFrame({
    "category":  [r.category        for r in reviews],
    "rating":    [r.rating          for r in reviews],
    "sentiment": sentiments.tolist(),
})
print("\nSentiment by category:")
display(
    df_sentiment.group_by("category").agg([
        pl.col("sentiment").mean().round(3).alias("avg_sentiment"),
        pl.col("sentiment").std().round(3).alias("std_sentiment"),
        pl.col("rating").mean().round(3).alias("avg_rating"),
        pl.len().alias("n_reviews"),
    ]).sort("avg_sentiment", descending=True)
)

---
## Step 4 — Embed review text with `nvidia/llama-nemotron-embed-1b-v2`

Each review (`title + body`) is embedded with the **`passage:` prefix** — this instructs
the Nemotron model to encode the text as a retrievable document rather than a query.
At search time, the query string gets the **`query:` prefix** instead.

This asymmetric dual-encoder design means semantically similar reviews surface even when
they use completely different words — e.g. *"broke after 2 months"* and *"stopped working
quickly"* land close together in the 1024-dim embedding space.

> **Note**: On CPU, embedding 20K reviews takes ~30–60 min. Reduce `MAX_REVIEWS_PER_CAT`
> to 2000 for a quick run, or use a GPU.

In [ ]:
def load_model(model_name: str, device: str):
    from transformers import AutoModel, AutoTokenizer
    import torch
    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    logging.info("Loading %s on %s ...", model_name, device)
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = (AutoModel.from_pretrained(
               model_name, trust_remote_code=True, torch_dtype=dtype)
           .to(device).eval())
    return tok, mdl


def encode_batch(
    texts: list[str], tokenizer, model, device: str,
    dim: int = EMBED_DIM, prefix: str = "passage",
) -> np.ndarray:
    import torch, torch.nn.functional as F
    prefixed = [f"{prefix}: {t[:512]}" for t in texts]
    batch = tokenizer(
        prefixed, padding=True, truncation=True,
        max_length=512, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        out = model(**batch)
    mask   = batch["attention_mask"]
    hidden = out.last_hidden_state.masked_fill(~mask[..., None].bool(), 0.0)
    emb    = hidden.sum(dim=1) / mask.sum(dim=1)[..., None]
    emb    = F.normalize(emb, dim=-1)
    vecs   = emb.float().cpu().numpy()[:, :dim]
    norms  = np.linalg.norm(vecs, axis=1, keepdims=True)
    return (vecs / np.clip(norms, 1e-12, None)).astype(np.float32)


tokenizer, model = load_model(EMBEDDING_MODEL, DEVICE)

review_embeddings: list[np.ndarray] = []
t0 = time.monotonic()

for i in range(0, len(review_texts), INFERENCE_BATCH):
    batch = review_texts[i : i + INFERENCE_BATCH]
    vecs  = encode_batch(batch, tokenizer, model, DEVICE)
    review_embeddings.append(vecs)
    if (i // INFERENCE_BATCH) % 25 == 0:
        pct  = (i + len(batch)) / len(review_texts) * 100
        rate = (i + len(batch)) / max(time.monotonic() - t0, 1e-6)
        print(f"  {i + len(batch):>7,}/{len(review_texts):,}  ({pct:.0f}%)  "
              f"{rate:.0f} reviews/s")

embeddings = np.vstack(review_embeddings)
elapsed    = time.monotonic() - t0

# Write embeddings back into AmazonReview objects
for review, vec in zip(reviews, embeddings):
    review.embedding = vec.tolist()

print(f"\nEmbedded {len(reviews):,} reviews in {elapsed:.1f}s  "
      f"({len(reviews)/elapsed:.0f} rev/s)  device={DEVICE}")
print(f"Matrix: {embeddings.shape}  dtype={embeddings.dtype}  "
      f"mean-norm={np.linalg.norm(embeddings, axis=1).mean():.4f}")

---
## Step 5 — VectorDB setup

**Sources**: `app/vectordb/milvus_backend.py`, `app/vectordb/elastic_backend.py`,
`app/vectordb/milvus_presets.py`

The `VectorDBClient` abstraction means swapping Milvus ↔ Elasticsearch requires
changing only one environment variable — all subsequent cells stay identical.

Default preset: **`gpu_cagra_cpu_adapted`** — builds with CAGRA/cuVS on GPU, serves
via HNSW-adapted CPU path for maximum recall on large corpora.

In [ ]:
from vectordb import create_vectordb_client
from vectordb.milvus_presets import available_milvus_presets

os.environ.update({
    "VECTORDB_BACKEND":    VECTORDB_BACKEND,
    "MILVUS_HOST":         MILVUS_HOST,
    "MILVUS_PORT":         str(MILVUS_PORT),
    "ELASTIC_HOST":        ELASTIC_HOST,
    "ELASTIC_PORT":        str(ELASTIC_PORT),
    "MILVUS_INDEX_PRESET": "gpu_cagra_cpu_adapted",
})

print(f"Available Milvus presets: {available_milvus_presets()}")

try:
    vdb = create_vectordb_client()
    vdb.connect()
    VDB_CONNECTED = True
    print(f"Connected to {VECTORDB_BACKEND}")

    if vdb.collection_exists(COLLECTION):
        vdb.drop_collection(COLLECTION)
        print(f"  Dropped existing '{COLLECTION}'")
    vdb.ensure_collection(COLLECTION, dim=EMBED_DIM)
    print(f"  Created '{COLLECTION}' (dim={EMBED_DIM}, preset=gpu_cagra_cpu_adapted)")

except Exception as e:
    print(f"VectorDB not available: {e}")
    print("  Start Milvus : docker compose --profile milvus up -d")
    print("  Start Elastic: docker compose --profile elastic up -d")
    VDB_CONNECTED = False
    vdb = None


---
## Step 6 — Ingest review embeddings

In [ ]:
def to_vdb_record(r: AmazonReview):
    """
    Adapter: AmazonReview → NewsArticle, used only at the VectorDB boundary.

    Milvus and Elasticsearch backends are built around the NewsArticle
    contract in app/vectordb/base.py. This is the single place in the
    notebook that touches that type — every other step uses AmazonReview.
    """
    from vectordb.base import NewsArticle
    return NewsArticle(
        id=r.id,
        headline=r.title,
        body=r.body,
        symbol=r.asin,
        timestamp=r.timestamp,
        sentiment_score=r.sentiment_score,
        embedding=r.embedding,
        cluster_id=r.cluster_id,
        cluster_label=r.cluster_label,
        x=r.x,
        y=r.y,
    )


if VDB_CONNECTED:
    total = 0
    t0 = time.monotonic()

    for i in range(0, len(reviews), INGEST_BATCH):
        batch   = [to_vdb_record(r) for r in reviews[i : i + INGEST_BATCH]]
        written = vdb.upsert_articles(COLLECTION, batch, flush=False)
        total  += written
        if (i // INGEST_BATCH) % 20 == 0:
            print(f"  {total:>7,} / {len(reviews):,} ingested")

    vdb.flush(COLLECTION)
    elapsed = time.monotonic() - t0
    print(f"\nDone: {total:,} reviews  in {elapsed:.1f}s  ({total/elapsed:.0f} rev/s)")
    print(f"Collection count: {vdb.count(COLLECTION):,}")
else:
    print("VDB not connected — search will use numpy cosine fallback.")

---
## Step 7 — Semantic search over customer reviews

Unlike keyword search, embedding-based search finds reviews that are **semantically
related** to the query even when they don't share the same words. For example:

- `"stopped working after a few months"` surfaces reviews about durability even if they
  say *"broke down"*, *"quit on me"*, *"dead after 3 months"*
- `"exceeded my expectations"` retrieves positive surprise reviews across all product types

Query vectors use the **`query:` prefix**; review vectors used **`passage:`** at ingest time.
When the VectorDB is running, results come from Milvus/Elasticsearch ANN search. When it
is offline, a numpy cosine similarity fallback runs directly over the `AmazonReview` objects.

In [ ]:
def _stars(sentiment: float) -> str:
    return "★" * int(round(sentiment * 2 + 3))


def semantic_search_vdb(query: str, *, top_k: int = 10) -> list:
    """Encode with 'query:' prefix and run ANN search via VectorDB."""
    q_vec   = encode_batch([query], tokenizer, model, DEVICE, prefix="query")[0].tolist()
    t0      = time.monotonic()
    results = vdb.vector_search(COLLECTION, q_vec, top_k=top_k)
    ms      = (time.monotonic() - t0) * 1000
    # VDB returns SearchResult(article=NewsArticle, score=float)
    # .headline maps back to AmazonReview.title (set in to_vdb_record)
    print(f"Query : '{query}'  [{ms:.0f}ms | top_k={top_k}]")
    for i, result in enumerate(results):
        r = result.article
        print(f"  [{i+1:2d}] score={result.score:.4f}  [{r.cluster_label}]  "
              f"{_stars(r.sentiment_score)}  {r.headline[:80]}")
    return results


def semantic_search_numpy(query: str, *, top_k: int = 10) -> list:
    """Numpy cosine fallback directly over AmazonReview objects — used when VDB is off."""
    q_vec   = encode_batch([query], tokenizer, model, DEVICE, prefix="query")[0]
    scores  = embeddings @ q_vec
    top_idx = np.argsort(scores)[::-1][:top_k]
    print(f"Query : '{query}'  [numpy cosine | top_k={top_k}]")
    for i, idx in enumerate(top_idx):
        r = reviews[idx]
        print(f"  [{i+1:2d}] score={scores[idx]:.4f}  [{r.category}]  "
              f"{_stars(r.sentiment_score)}  {r.title[:80]}")
    return [(reviews[i], scores[i]) for i in top_idx]


search_fn = semantic_search_vdb if VDB_CONNECTED else semantic_search_numpy

# Review-specific queries — surface real customer pain points and praise
QUERIES = [
    "stopped working after a few months, terrible quality",
    "battery drains too fast, disappointing",
    "easy to set up, works exactly as described",
    "great value for the price, highly recommend",
    "shipping was damaged and packaging broken",
    "too small, sizing runs very small",
    "noisy compressor keeps me awake at night",
    "excellent customer service resolved my issue quickly",
]

for query in QUERIES:
    search_fn(query, top_k=5)
    print()

In [ ]:
# Category-specific queries
print("=== Category-specific semantic search ===")
for cat, query in [
    ("Appliances",             "energy consumption too high, electricity bill increased"),
    ("Electronics",            "screen flickering and display issues after update"),
    ("Clothing_Shoes_Jewelry", "material feels cheap and itchy, not as pictured"),
]:
    if any(r.category == cat for r in reviews):
        search_fn(query, top_k=5)
    else:
        print(f"Category '{cat}' not in loaded data — skipping")
    print()

---
## Step 8 — Sentiment timeseries with Polars

We aggregate sentiment (derived from star ratings) over time and by category using Polars
`group_by_dynamic` — binning reviews by month without any date truncation UDFs. This is
the same aggregation pattern used on the `/api/correlation` endpoint in the production app.

In [ ]:
df_ts = pl.DataFrame({
    "category":  [r.category        for r in reviews],
    "timestamp": [r.timestamp       for r in reviews],
    "rating":    [r.rating          for r in reviews],
    "sentiment": [r.sentiment_score for r in reviews],
    "asin":      [r.asin            for r in reviews],
}).with_columns(
    pl.col("timestamp").str.to_datetime(time_unit="us", strict=False).alias("dt")
).filter(pl.col("dt").is_not_null())

# Monthly sentiment timeseries per category
monthly = (
    df_ts
    .sort("dt")
    .group_by_dynamic("dt", every="1mo", group_by="category")
    .agg([
        pl.col("sentiment").mean().round(3).alias("avg_sentiment"),
        pl.col("sentiment").count().alias("review_count"),
    ])
    .sort(["category", "dt"])
)
print("Monthly sentiment (sample):")
display(monthly.head(12))

# Sentiment by rating tier
print("\nSentiment distribution by rating:")
display(
    df_ts.with_columns(pl.col("rating").cast(pl.Int32).alias("stars"))
    .group_by(["category", "stars"]).agg([
        pl.col("sentiment").mean().round(3).alias("avg_sentiment"),
        pl.len().alias("n_reviews"),
    ]).sort(["category", "stars"])
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

cats_available = monthly["category"].unique().to_list()
fig, axes = plt.subplots(len(cats_available), 1,
                          figsize=(13, 4 * len(cats_available)), sharex=False)
if len(cats_available) == 1:
    axes = [axes]

for ax, cat in zip(axes, sorted(cats_available)):
    cat_data = monthly.filter(pl.col("category") == cat).sort("dt")
    dts    = cat_data["dt"].to_list()
    sents  = cat_data["avg_sentiment"].to_list()
    counts = cat_data["review_count"].to_list()

    ax2 = ax.twinx()
    ax2.bar(dts, counts, color="steelblue", alpha=0.25, label="Review count")
    ax2.set_ylabel("Reviews / month", color="steelblue", fontsize=9)

    colors = ["green" if s >= 0 else "red" for s in sents]
    ax.plot(dts, sents, color="black", linewidth=1.5, zorder=3)
    ax.fill_between(dts, 0, sents, where=[s >= 0 for s in sents],
                    color="green", alpha=0.3, zorder=2)
    ax.fill_between(dts, 0, sents, where=[s < 0 for s in sents],
                    color="red", alpha=0.3, zorder=2)
    ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylabel("Avg Sentiment", fontsize=9)
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(f"{cat} — Monthly Review Sentiment")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.grid(alpha=0.2)

plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(DATA_DIR / "monthly_sentiment.png", dpi=120)
plt.show()


In [ ]:
# Export monthly sentiment for the Streamlit dashboard
# (reference/frontend/amazon_app.py — Sentiment Timeline tab)
SENTIMENT_PARQUET = DATA_DIR / "amazon_reviews_sentiment.parquet"
monthly.write_parquet(SENTIMENT_PARQUET)
print(f"Saved sentiment data -> {SENTIMENT_PARQUET}")

---
## Step 9 — BERTopic-style clustering: UMAP + HDBSCAN

Project 1024-dim review embeddings to 2D with UMAP, then cluster with HDBSCAN.
The resulting clusters correspond to **review topic groups** — emerging purely from the
semantic structure of the review text, with no labels provided. Examples:

- Durability / reliability complaints
- Sizing / fit issues (Clothing)
- Setup difficulty / instructions
- Exceptional value / price praise
- Delivery and packaging problems

Each cluster is then labeled automatically using the most frequent non-stop words found
in its review text.

GPU path: **RAPIDS cuML UMAP** — falls back to **umap-learn** on CPU if unavailable.

In [ ]:
MAX_CLUSTER_POINTS = min(len(embeddings), 8_000)
rng_sample = np.random.default_rng(42)
idx_sample = rng_sample.choice(len(embeddings), MAX_CLUSTER_POINTS, replace=False)

emb_sample = embeddings[idx_sample]
rev_sample = [reviews[i] for i in idx_sample]    # AmazonReview objects
cat_sample = [r.category for r in rev_sample]

n_neighbors = min(15, max(2, len(emb_sample) - 1))

print(f"Running UMAP on {len(emb_sample):,} reviews (dim={emb_sample.shape[1]}) ...")
t0 = time.monotonic()
gpu_umap = False

if GPU_AVAILABLE:
    try:
        from cuml import UMAP as cuUMAP
        reducer  = cuUMAP(n_components=2, n_neighbors=n_neighbors,
                          min_dist=0.05, metric="cosine",
                          random_state=42, output_type="numpy")
        coords   = reducer.fit_transform(emb_sample)
        gpu_umap = True
        print(f"UMAP: cuML GPU  ({(time.monotonic()-t0)*1000:.0f}ms)")
    except Exception as e:
        print(f"cuML unavailable ({e}) — falling back to CPU")

if not gpu_umap:
    from umap import UMAP
    reducer = UMAP(n_components=2, n_neighbors=n_neighbors,
                   min_dist=0.05, metric="cosine", random_state=42)
    coords  = reducer.fit_transform(emb_sample)
    print(f"UMAP: umap-learn CPU  ({(time.monotonic()-t0)*1000:.0f}ms)")

print(f"UMAP output: {coords.shape}")

In [ ]:
try:
    import hdbscan
    min_cluster = max(10, len(emb_sample) // 80)
    clusterer   = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    t0 = time.monotonic()
    cluster_labels = clusterer.fit_predict(coords)
    ms = (time.monotonic() - t0) * 1000
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise    = (cluster_labels == -1).sum()
    print(f"HDBSCAN: {n_clusters} clusters  {n_noise} noise  ({ms:.0f}ms)")

    # ── Label each cluster by its most frequent review keywords ───────────────
    import re
    STOP_WORDS = {
        "the", "a", "an", "is", "it", "i", "my", "me", "this", "that", "was",
        "are", "be", "to", "of", "and", "for", "in", "on", "with", "at", "by",
        "have", "had", "has", "not", "but", "so", "just", "very", "as", "or",
        "do", "did", "its", "one", "from", "they", "them", "we", "he", "she",
        "up", "get", "got", "than", "when", "if", "out", "about", "no", "what",
        "you", "your", "all", "can", "after", "our", "will", "would", "which",
        "been", "were", "much", "how", "some", "also", "well", "use", "used",
    }

    cluster_label_map: dict[int, str] = {}
    for cid in sorted(set(cluster_labels)):
        if cid < 0:
            continue
        mask  = cluster_labels == cid
        texts = [rev_sample[i].body[:200] for i, m in enumerate(mask) if m]
        words = re.findall(r"[a-z]+", " ".join(texts).lower())
        freq  = Counter(w for w in words if w not in STOP_WORDS and len(w) > 3)
        top   = [w for w, _ in freq.most_common(4)]
        cluster_label_map[cid] = ", ".join(top) or f"Cluster {cid}"

    # Write cluster results back into AmazonReview objects
    for i, (rev, cid) in enumerate(zip(rev_sample, cluster_labels)):
        rev.cluster_id    = int(cid)
        rev.cluster_label = cluster_label_map.get(int(cid), rev.category)
        rev.x = float(coords[i, 0])
        rev.y = float(coords[i, 1])

    print("\nDiscovered review topic clusters:")
    for cid, label in sorted(cluster_label_map.items()):
        n = (cluster_labels == cid).sum()
        cat_votes = Counter(cat_sample[i] for i, m in enumerate(cluster_labels == cid) if m)
        dominant  = cat_votes.most_common(1)[0][0] if cat_votes else "?"
        print(f"  [{cid:3d}] {n:5d} reviews  [{dominant}]  → {label}")

except ImportError:
    print("hdbscan not installed (pip install hdbscan) — using category as cluster labels")
    unique_cats  = sorted(set(cat_sample))
    cat_to_id    = {c: i for i, c in enumerate(unique_cats)}
    cluster_labels     = np.array([cat_to_id[c] for c in cat_sample])
    cluster_label_map  = {i: c for c, i in cat_to_id.items()}
    n_clusters = len(unique_cats)

---
## Step 10 — UMAP scatter visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

unique_cids = sorted(set(cluster_labels))
palette     = cm.get_cmap("tab20", max(len(unique_cids), 1))
engine_lbl  = "cuML GPU" if gpu_umap else "umap-learn CPU"

fig, ax = plt.subplots(figsize=(14, 10))

# Noise
noise_mask = cluster_labels == -1
if noise_mask.any():
    ax.scatter(coords[noise_mask, 0], coords[noise_mask, 1],
               c="#cccccc", s=5, alpha=0.2, label="Noise", zorder=1)

# Clusters
for i, cid in enumerate(c for c in unique_cids if c >= 0):
    mask  = cluster_labels == cid
    label = cluster_label_map.get(cid, f"Cluster {cid}")
    color = palette(i % 20)
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=[color], s=8, alpha=0.55, label=label[:45], zorder=2)
    cx, cy = coords[mask, 0].mean(), coords[mask, 1].mean()
    ax.annotate(
        label[:35], (cx, cy), fontsize=7, ha="center", va="center", zorder=3,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="none"),
    )

ax.set_title(
    f"Amazon Reviews 2023 — Review Topic Clusters\n"
    f"UMAP ({engine_lbl}) + HDBSCAN  ·  {len(emb_sample):,} reviews  ·  "
    f"{n_clusters} topic clusters  ·  Nemotron {EMBED_DIM}D"
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=7, framealpha=0.9)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(DATA_DIR / "review_topic_clusters.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

sents_sample = np.array([r.sentiment_score for r in rev_sample])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Left: sentiment heatmap
sc = ax1.scatter(coords[:, 0], coords[:, 1],
                 c=sents_sample, cmap="RdYlGn",
                 vmin=-1.0, vmax=1.0, s=7, alpha=0.65)
cbar = fig.colorbar(sc, ax=ax1, pad=0.01)
cbar.set_label("Sentiment (from star rating)")
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.set_ticklabels(["1★", "2★", "3★", "4★", "5★"])
ax1.set_title("Coloured by Customer Sentiment (Star Rating)")
ax1.set_xlabel("UMAP-1")
ax1.set_ylabel("UMAP-2")
ax1.grid(alpha=0.15)

# Right: coloured by source category
cat_unique = sorted(set(cat_sample))
cat_palette = cm.get_cmap("Set1", len(cat_unique))
cat_color_map = {c: cat_palette(i) for i, c in enumerate(cat_unique)}
cat_colors = [cat_color_map[c] for c in cat_sample]

ax2.scatter(coords[:, 0], coords[:, 1], c=cat_colors, s=7, alpha=0.55)
for cat, color in cat_color_map.items():
    ax2.scatter([], [], c=[color], label=cat, s=30)
ax2.set_title("Coloured by Product Category")
ax2.set_xlabel("UMAP-1")
ax2.set_ylabel("UMAP-2")
ax2.legend(fontsize=9, framealpha=0.9)
ax2.grid(alpha=0.15)

plt.suptitle(f"Amazon Reviews 2023 — Embedding Space  ({engine_lbl})",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "review_embedding_space.png", dpi=130, bbox_inches="tight")
plt.show()

---
## Cleanup

In [ ]:
if VDB_CONNECTED and vdb is not None:
    # vdb.drop_collection(COLLECTION)  # uncomment to free Milvus storage
    vdb.disconnect()
    print("Disconnected from VectorDB.")


---
## Summary

| Step | Tool(s) | What happened |
|------|---------|---------------|
| 1 | `gzip` + `json` | Streamed `Appliances.jsonl.gz`, `Electronics.jsonl.gz`, `Clothing_Shoes_and_Jewelry.jsonl.gz` from amazon-reviews-2023.github.io |
| 2 | Polars + matplotlib | EDA: rating distributions, review length, category volume |
| 3 | `AmazonReview` dataclass | Domain-appropriate record with `title`, `body`, `asin`, `category`, `rating`, `sentiment_score` |
| 4 | **Nemotron** (`transformers`, `torch`) | `passage: <title>. <body>` → 1024-dim L2-normed vectors in `AmazonReview.embedding` |
| 5 | **Milvus + cuVS** / **Elasticsearch** | `GPU_CAGRA` ANN collection via `VectorDBClient` abstraction |
| 6 | `to_vdb_record()` + `upsert_articles` | Thin adapter converts `AmazonReview → NewsArticle` **at VectorDB boundary only**; deferred WAL flush |
| 7 | `vector_search` + numpy fallback | `query: <complaint/praise>` → top-k semantically similar reviews |
| 8 | **Polars** `group_by_dynamic` | Monthly sentiment timeseries + rating-tier breakdown |
| 9 | **RAPIDS cuML UMAP** / **umap-learn** + **HDBSCAN** | 1024D → 2D; keyword-labeled review topic clusters written back into `AmazonReview` |
| 10 | matplotlib | Topic cluster scatter + sentiment/category overlay |

### Design note: `AmazonReview` vs `NewsArticle`

The entire notebook works with `AmazonReview` — a dataclass defined here whose fields
match the review domain (`title`, `body`, `asin`, `category`, `rating`). The only exception
is `to_vdb_record()` in Step 6, which adapts `AmazonReview` into the `NewsArticle` type
that the Milvus/Elasticsearch backends require. This keeps the domain model clean while
still reusing the production VectorDB infrastructure unchanged.

**See also**:
- `03_structured_amazon_products.ipynb` — structured product catalog analytics (Kaggle)
- `02_unstructured_pipeline.ipynb` — same pipeline on synthetic GDELT news